# Логистическая модель и функция потерь

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Переводить линейный счёт в вероятность и вычислять логистическую потерю.

## Число на прямой и вероятность

Линейный счёт z=b+w·x может быть любым числом. **Сигмоида** σ(z)=1/(1+e^(−z)) превращает его в число между 0 и 1. При z=0 получается 0.5, при большом положительном z — число близко к 1, при отрицательном — к 0. Логистическая модель использует σ(z) как оценку вероятности положительного класса.

Название «логистическая регрессия» не означает прогноз произвольного числового ответа: стандартное применение — классификация. Вероятность и метка — разные выходы. Метка получается после выбора порога, например p≥0.5.

## Разобранный пример: уверенность и ошибка

Если z=ln3, e^(−z)=1/3, поэтому p=3/4. Для правильной метки 1 штраф равен −ln0.75≈0.288; для правильной метки 0 штраф −ln0.25≈1.386. Одинаковый прогноз штрафуется по-разному в зависимости от реального ответа.

Для y∈{0,1} единая формула $\ell=-y\ln p-(1-y)\ln(1-p)$. Её производная по линейному счёту равна p−y: при y=1 и p=0.75 градиент −0.25 побуждает увеличить счёт. Веса меняют ещё с учётом признаков по правилу цепочки.

Покажем промежуточные производные. Для сигмоиды правило цепочки даёт $p^{\prime}(z)=e^{-z}/(1+e^{-z})^2=p(1-p)$. Производная штрафа по p равна $-y/p+(1-y)/(1-p)=(p-y)/(p(1-p))$. Умножаем её на $p^{\prime}(z)$ и получаем $p-y$. При конечном z сигмоида строго между 0 и 1, поэтому эти знаменатели ненулевые. По весу $w_j$ получается $(p-y)x_j$, по свободному члену — $p-y$.

Обучение минимизирует средний штраф, часто с регуляризацией. Вероятностная интерпретация опирается на модель и качество данных; число 0.8 не становится автоматически хорошо откалиброванным шансом. В sklearn метод predict_proba выдаёт столбцы вероятностей в порядке classes_, а predict — готовые метки.

Для вычислений сигмоиды на очень больших по модулю числах нужна устойчивая реализация. В ручном примере используем умеренные числа; библиотечную модель не заменяем прямым экспоненциальным выражением без контроля переполнения.

Параметр `C` в библиотечной логистической модели задаёт обратную силу регуляризации: большее значение ослабляет штраф. В этом примере он фиксирован, а не подбирается по запросам.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(C=1., random_state=17).fit([[-2.], [-1.], [1.], [2.]], [0, 0, 1, 1])
probabilities = model.predict_proba([[0.], [2.]])
assert model.classes_.tolist() == [0, 1]
assert probabilities.shape == (2, 2)
assert np.allclose(probabilities.sum(axis=1), 1)
assert probabilities[1, 1] > probabilities[0, 1]
print(probabilities)


## Типичные ошибки

Не передавайте вероятности как метки в accuracy. Не путайте столбцы predict_proba: сверяйте classes_.

## Задачи

### 1. Базовая

Чему равна σ(0)?

<details>
<summary>Подсказка</summary>

e⁰=1.

</details>



### 2. Базовая

Что выдаёт predict_proba?

<details>
<summary>Подсказка</summary>

Это не один окончательный класс.

</details>



### 3. Прикладная

Для z=ln3 найдите p.

<details>
<summary>Подсказка</summary>

Подставьте e^(−ln3)=1/3.

</details>



### 4. Прикладная

Для p=0.8,y=1 найдите градиент по z.

<details>
<summary>Подсказка</summary>

Используйте p−y.

</details>



### 5. Олимпиадная

Могут ли две модели иметь одинаковые метки и разные логистические потери?

<details>
<summary>Подсказка</summary>

Вероятности могут лежать по одну сторону порога.

</details>



### 6. Олимпиадная

Почему результат 0.8 нельзя без проверки трактовать как частоту 80%?

<details>
<summary>Подсказка</summary>

Нужна калибровка на подходящих новых объектах.

</details>



## Итоги и самопроверка

Вычислите штраф для обеих меток при одном p. Объясните отличие обучения вероятности от выбора порога.

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

1/2.

### Задача 2

Вероятности классов, порядок которых указан в classes_.

### Задача 3

p=3/4.

### Задача 4

−0.2; шаг против градиента увеличивает z.

### Задача 5

Да. p=0.6 и p=0.9 оба дают класс 1, но при y=1 второй штраф меньше.

### Задача 6

Численный выход модели может быть систематически слишком уверенным или не соответствовать новому распределению.